In [3]:
install.packages(c("readxl", "writexl", "jsonlite", "dplyr", "DBI", "RSQLite", "lubridate"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [4]:
library(readxl)
library(writexl)
library(jsonlite)
library(dplyr)
library(DBI)
library(RSQLite)
library(lubridate)


Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [20]:
stopifnot(file.exists("Online Retail.xlsx"))
raw <- read_excel("Online Retail.xlsx")

names(raw) <- trimws(names(raw))
print(names(raw))
print(dim(raw))

[1] "InvoiceNo"   "StockCode"   "Description" "Quantity"    "InvoiceDate"
[6] "UnitPrice"   "CustomerID"  "Country"    
[1] 541909      8


In [22]:
head(raw)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


In [23]:
str(raw)

tibble [541,909 × 8] (S3: tbl_df/tbl/data.frame)
 $ InvoiceNo  : chr [1:541909] "536365" "536365" "536365" "536365" ...
 $ StockCode  : chr [1:541909] "85123A" "71053" "84406B" "84029G" ...
 $ Description: chr [1:541909] "WHITE HANGING HEART T-LIGHT HOLDER" "WHITE METAL LANTERN" "CREAM CUPID HEARTS COAT HANGER" "KNITTED UNION FLAG HOT WATER BOTTLE" ...
 $ Quantity   : num [1:541909] 6 6 8 6 6 2 6 6 6 32 ...
 $ InvoiceDate: POSIXct[1:541909], format: "2010-12-01 08:26:00" "2010-12-01 08:26:00" ...
 $ UnitPrice  : num [1:541909] 2.55 3.39 2.75 3.39 3.39 7.65 4.25 1.85 1.85 1.69 ...
 $ CustomerID : num [1:541909] 17850 17850 17850 17850 17850 ...
 $ Country    : chr [1:541909] "United Kingdom" "United Kingdom" "United Kingdom" "United Kingdom" ...


In [24]:
colSums(is.na(raw))

InvoiceNo   StockCode Description    Quantity InvoiceDate   UnitPrice 
          0           0        1454           0           0           0 
 CustomerID     Country 
     135080           0

In [25]:
sum(duplicated(raw))

[1] 5268

In [26]:
sum(raw$Quantity <= 0, na.rm = TRUE)

[1] 10624

In [27]:
clean_data <- raw %>%
  filter(
    !is.na(InvoiceNo),
    !is.na(StockCode),
    !is.na(Quantity),
    !is.na(UnitPrice),
    !is.na(CustomerID),
    !is.na(Country),
    Quantity > 0,
    UnitPrice > 0
  ) %>%
  distinct()

In [28]:
clean_data <- clean_data %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

head(clean_data)

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>,<dbl>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,15.30


In [29]:
print(dim(clean_data))

summary(clean_data)

[1] 392692      9


     InvoiceNo          StockCode         Description        Quantity       
 Length   :392692   Length   :392692   Length   :392692   Min.   :    1.00  
 N.unique : 18532   N.unique :  3665   N.unique :  3866   1st Qu.:    2.00  
 N.blank  :     0   N.blank  :     0   N.blank  :     0   Median :    6.00  
 Min.nchar:     6   Min.nchar:     1   Min.nchar:     6   Mean   :   13.12  
 Max.nchar:     6   Max.nchar:    12   Max.nchar:    35   3rd Qu.:   12.00  
                                                          Max.   :80995.00  
  InvoiceDate                    UnitPrice          CustomerID   
 Min.   :2010-12-01 08:26:00   Min.   :   0.001   Min.   :12346  
 1st Qu.:2011-04-07 11:12:00   1st Qu.:   1.250   1st Qu.:13955  
 Median :2011-07-31 12:02:00   Median :   1.950   Median :15150  
 Mean   :2011-07-10 19:13:07   Mean   :   3.126   Mean   :15288  
 3rd Qu.:2011-10-20 12:53:00   3rd Qu.:   3.750   3rd Qu.:16791  
 Max.   :2011-12-09 12:50:00   Max.   :8142.750   Max.   :18287  

In [30]:
colSums(is.na(clean_data))

InvoiceNo   StockCode Description    Quantity InvoiceDate   UnitPrice 
          0           0           0           0           0           0 
 CustomerID     Country     Revenue 
          0           0           0

In [31]:
transactions <- clean_data %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate
  )

write.csv(
  transactions,
  "transactions.csv",
  row.names = FALSE
)

head(transactions)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00
536365,22752,17850,2,2010-12-01 08:26:00


In [32]:
products <- clean_data %>%
  select(
    StockCode,
    Description,
    UnitPrice
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

write_json(
  products,
  "products.json",
  pretty = TRUE,
  auto_unbox = TRUE
)

head(products)

StockCode,Description,UnitPrice
<chr>,<chr>,<dbl>
85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
71053,WHITE METAL LANTERN,3.39
84406B,CREAM CUPID HEARTS COAT HANGER,2.75
84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
22752,SET 7 BABUSHKA NESTING BOXES,7.65


In [33]:
customers <- clean_data %>%
  select(
    CustomerID,
    Country
  ) %>%
  distinct(CustomerID, .keep_all = TRUE)

write_xlsx(
  customers,
  "customers.xlsx"
)

head(customers)

CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


In [34]:
transactions_imported <- read.csv("transactions.csv")

products_imported <- fromJSON("products.json")

customers_imported <- read_excel("customers.xlsx")

print(dim(transactions_imported))
print(dim(products_imported))
print(dim(customers_imported))

[1] 392692      5
[1] 3665    3
[1] 4338    2


In [35]:
head(transactions_imported)

head(products_imported)

head(customers_imported)

,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
,<int>,<chr>,<int>,<int>,<chr>
1,536365,85123A,17850,6,2010-12-01 08:26:00
2,536365,71053,17850,6,2010-12-01 08:26:00
3,536365,84406B,17850,8,2010-12-01 08:26:00
4,536365,84029G,17850,6,2010-12-01 08:26:00
5,536365,84029E,17850,6,2010-12-01 08:26:00
6,536365,22752,17850,2,2010-12-01 08:26:00


,StockCode,Description,UnitPrice
,<chr>,<chr>,<dbl>
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,71053,WHITE METAL LANTERN,3.39
3,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39
6,22752,SET 7 BABUSHKA NESTING BOXES,7.65


CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom
15291,United Kingdom


In [36]:
names(transactions_imported)
names(products_imported)
names(customers_imported)

[1] "InvoiceNo"   "StockCode"   "CustomerID"  "Quantity"    "InvoiceDate"

[1] "StockCode"   "Description" "UnitPrice"

[1] "CustomerID" "Country"

In [37]:
sales_products <- transactions_imported %>%
  left_join(
    products_imported,
    by = "StockCode"
  )

print(dim(sales_products))

head(sales_products)

[1] 392692      7


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice
,<int>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>
1,536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39
3,536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75
4,536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39
6,536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65


In [38]:
final_data <- sales_products %>%
  left_join(
    customers_imported,
    by = "CustomerID"
  )

print(dim(final_data))

head(final_data)

[1] 392692      8


,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country
,<int>,<chr>,<dbl>,<int>,<chr>,<chr>,<dbl>,<chr>
1,536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom
2,536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,United Kingdom
3,536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom
4,536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom
5,536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom
6,536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,United Kingdom


In [39]:
final_data <- final_data %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

head(final_data)

,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
,<int>,<chr>,<dbl>,<int>,<chr>,<chr>,<dbl>,<chr>,<dbl>
1,536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
2,536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,United Kingdom,20.34
3,536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
4,536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
5,536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34
6,536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,United Kingdom,15.30


In [40]:
print(dim(final_data))

print(names(final_data))

summary(final_data)

[1] 392692      9
[1] "InvoiceNo"   "StockCode"   "CustomerID"  "Quantity"    "InvoiceDate"
[6] "Description" "UnitPrice"   "Country"     "Revenue"    


   InvoiceNo          StockCode        CustomerID       Quantity       
 Min.   :536365   Length   :392692   Min.   :12346   Min.   :    1.00  
 1st Qu.:549234   N.unique :  3665   1st Qu.:13955   1st Qu.:    2.00  
 Median :561874   N.blank  :     0   Median :15150   Median :    6.00  
 Mean   :560591   Min.nchar:     1   Mean   :15288   Mean   :   13.12  
 3rd Qu.:572061   Max.nchar:    12   3rd Qu.:16791   3rd Qu.:   12.00  
 Max.   :581587                      Max.   :18287   Max.   :80995.00  
    InvoiceDate        Description       UnitPrice            Country      
 Length   :392692   Length   :392692   Min.   :  0.001   Length   :392692  
 N.unique : 17282   N.unique :  3639   1st Qu.:  1.250   N.unique :    37  
 N.blank  :     0   N.blank  :     0   Median :  1.950   N.blank  :     0  
 Min.nchar:    19   Min.nchar:     6   Mean   :  2.955   Min.nchar:     3  
 Max.nchar:    19   Max.nchar:    35   3rd Qu.:  3.750   Max.nchar:    20  
                                       M

In [41]:
unmatched_products <- final_data %>%
  filter(is.na(Description))

print(nrow(unmatched_products))

[1] 0


In [42]:
unmatched_customers <- final_data %>%
  filter(is.na(Country))

print(nrow(unmatched_customers))

[1] 0


A left_join() was used because all valid transaction records should be
retained in the final dataset. Product and customer information is added
when a matching StockCode or CustomerID is available. This also allows
unmatched product and customer records to be identified.

In [43]:
total_revenue <- final_data %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE)
  )

print(total_revenue)

  Total_Revenue
1       9546219


In [44]:
top_5_products <- final_data %>%
  group_by(StockCode, Description) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

print(top_5_products)

# A tibble: 5 × 3
  StockCode Description                        Total_Revenue
  <chr>     <chr>                                      <dbl>
1 23843     PAPER CRAFT , LITTLE BIRDIE              168470.
2 22423     REGENCY CAKESTAND 3 TIER                 135495.
3 85123A    WHITE HANGING HEART T-LIGHT HOLDER        93746.
4 23166     MEDIUM CERAMIC TOP STORAGE JAR            81033.
5 85099B    JUMBO BAG RED RETROSPOT                   76029.


In [45]:
top_5_countries <- final_data %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue)) %>%
  slice_head(n = 5)

print(top_5_countries)

# A tibble: 5 × 2
  Country        Total_Revenue
  <chr>                  <dbl>
1 United Kingdom      7868409.
2 Netherlands          329131.
3 EIRE                 287261.
4 Germany              233804.
5 France               203628.


In [46]:
top_5_customers <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Purchase_Value)) %>%
  slice_head(n = 5)

print(top_5_customers)

# A tibble: 5 × 2
  CustomerID Total_Purchase_Value
       <dbl>                <dbl>
1      18102              383153.
2      14646              323768.
3      17450              171052.
4      16446              168472.
5      14911              155092.


In [47]:
customer_value <- final_data %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase_Value = sum(Revenue, na.rm = TRUE),
    .groups = "drop"
  )

head(customer_value)

CustomerID,Total_Purchase_Value
<dbl>,<dbl>
12346,77183.60
12347,4737.58
12348,1661.64
12349,1523.47
12350,307.48
12352,1444.07


In [48]:
q25 <- quantile(
  customer_value$Total_Purchase_Value,
  0.25,
  na.rm = TRUE
)

q50 <- quantile(
  customer_value$Total_Purchase_Value,
  0.50,
  na.rm = TRUE
)

q75 <- quantile(
  customer_value$Total_Purchase_Value,
  0.75,
  na.rm = TRUE
)

print(q25)
print(q50)
print(q75)

    25% 
317.835 
   50% 
703.57 
     75% 
1744.907 


In [49]:
customer_value <- customer_value %>%
  mutate(
    Customer_Category = case_when(
      Total_Purchase_Value <= q25 ~ "Low Value",
      Total_Purchase_Value <= q50 ~ "Medium Value",
      Total_Purchase_Value <= q75 ~ "High Value",
      TRUE ~ "Premium"
    )
  )

head(customer_value)

CustomerID,Total_Purchase_Value,Customer_Category
<dbl>,<dbl>,<chr>
12346,77183.60,Premium
12347,4737.58,Premium
12348,1661.64,High Value
12349,1523.47,High Value
12350,307.48,Low Value
12352,1444.07,High Value


In [50]:
customer_category_summary <- customer_value %>%
  count(Customer_Category)

print(customer_category_summary)

# A tibble: 4 × 2
  Customer_Category     n
  <chr>             <int>
1 High Value         1084
2 Low Value          1085
3 Medium Value       1084
4 Premium            1085


In [51]:
final_data <- final_data %>%
  left_join(
    customer_value %>%
      select(CustomerID, Customer_Category),
    by = "CustomerID"
  )

head(final_data)

,InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue,Customer_Category
,<int>,<chr>,<dbl>,<int>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1,536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30,Premium
2,536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,United Kingdom,20.34,Premium
3,536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00,Premium
4,536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34,Premium
5,536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34,Premium
6,536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,United Kingdom,15.30,Premium


In [52]:
country_performance <- final_data %>%
  group_by(Country) %>%
  summarise(
    Total_Revenue = sum(Revenue, na.rm = TRUE),
    Transaction_Count = n(),
    .groups = "drop"
  ) %>%
  arrange(desc(Total_Revenue))

print(country_performance)

# A tibble: 37 × 3
   Country        Total_Revenue Transaction_Count
   <chr>                  <dbl>             <int>
 1 United Kingdom      7868409.            349203
 2 Netherlands          329131.              2359
 3 EIRE                 287261.              7226
 4 Germany              233804.              9025
 5 France               203628.              8326
 6 Australia            157316.              1253
 7 Spain                 62081.              2408
 8 Switzerland           57268.              1825
 9 Belgium               43099.              2006
10 Japan                 42083.               321
# ℹ 27 more rows


In [53]:
high_performing_market <- country_performance %>%
  slice_head(n = 1)

print(high_performing_market)

# A tibble: 1 × 3
  Country        Total_Revenue Transaction_Count
  <chr>                  <dbl>             <int>
1 United Kingdom      7868409.            349203


In [54]:
underperforming_market <- country_performance %>%
  filter(Transaction_Count >= 10) %>%
  arrange(Total_Revenue) %>%
  slice_head(n = 1)

print(underperforming_market)

# A tibble: 1 × 3
  Country Total_Revenue Transaction_Count
  <chr>           <dbl>             <int>
1 Bahrain          545.                17


In [55]:
cat(
  "High-performing market:",
  high_performing_market$Country,
  "\nRevenue:",
  round(high_performing_market$Total_Revenue, 2),
  "\n\n"
)

cat(
  "Underperforming market:",
  underperforming_market$Country,
  "\nRevenue:",
  round(underperforming_market$Total_Revenue, 2),
  "\n\n"
)

cat(
  "Top product:",
  top_5_products$Description[1],
  "\nRevenue:",
  round(top_5_products$Total_Revenue[1], 2),
  "\n\n"
)

cat(
  "Top customer:",
  top_5_customers$CustomerID[1],
  "\nPurchase value:",
  round(top_5_customers$Total_Purchase_Value[1], 2)
)

High-performing market: United Kingdom 
Revenue: 7868409 

Underperforming market: Bahrain 
Revenue: 544.8 

Top product: PAPER CRAFT , LITTLE BIRDIE 
Revenue: 168469.6 

Top customer: 18102 
Purchase value: 383153

In [56]:
library(DBI)
library(RSQLite)

In [57]:
db <- dbConnect(
  SQLite(),
  "retail_sales.db"
)

print(db)

<SQLiteConnection>
  Path: retail_sales.db
  Extensions: TRUE


In [58]:
dbWriteTable(
  db,
  "retail_sales",
  final_data,
  overwrite = TRUE
)

print(dbListTables(db))

[1] "retail_sales"


In [59]:
dbGetQuery(
  db,
  "SELECT COUNT(*) AS Total_Rows FROM retail_sales"
)

Total_Rows
<int>
392692


In [60]:
top_customers_sql <- dbGetQuery(
  db,
  "
  SELECT
      CustomerID,
      ROUND(SUM(Revenue), 2) AS Total_Revenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY Total_Revenue DESC
  LIMIT 5
  "
)

print(top_customers_sql)

  CustomerID Total_Revenue
1      18102      383153.0
2      14646      323767.5
3      17450      171051.9
4      16446      168472.5
5      14911      155092.0


In [61]:
country_revenue_sql <- dbGetQuery(
  db,
  "
  SELECT
      Country,
      ROUND(SUM(Revenue), 2) AS Total_Revenue
  FROM retail_sales
  GROUP BY Country
  ORDER BY Total_Revenue DESC
  "
)

print(country_revenue_sql)

                Country Total_Revenue
1        United Kingdom    7868408.65
2           Netherlands     329130.77
3                  EIRE     287260.87
4               Germany     233804.06
5                France     203628.28
6             Australia     157316.33
7                 Spain      62080.85
8           Switzerland      57268.14
9               Belgium      43098.99
10                Japan      42082.88
11               Sweden      37803.37
12               Norway      35357.24
13             Portugal      29159.14
14              Finland      20765.93
15      Channel Islands      20360.83
16              Denmark      19771.33
17                Italy      17298.96
18               Cyprus      14400.47
19            Singapore      10106.14
20              Austria       8871.02
21               Israel       7941.02
22               Poland       7913.05
23               Greece       5101.78
24              Iceland       4737.58
25               Canada       3608.40
26          

In [62]:
top_products_sql <- dbGetQuery(
  db,
  "
  SELECT
      StockCode,
      Description,
      ROUND(SUM(Revenue), 2) AS Total_Revenue
  FROM retail_sales
  GROUP BY StockCode, Description
  ORDER BY Total_Revenue DESC
  LIMIT 5
  "
)

print(top_products_sql)

  StockCode                        Description Total_Revenue
1     23843        PAPER CRAFT , LITTLE BIRDIE     168469.60
2     22423           REGENCY CAKESTAND 3 TIER     135495.30
3    85123A WHITE HANGING HEART T-LIGHT HOLDER      93745.65
4     23166     MEDIUM CERAMIC TOP STORAGE JAR      81032.64
5    85099B            JUMBO BAG RED RETROSPOT      76028.70


In [63]:
top_country <- country_revenue_sql[1, ]

cat(
  "Business Insight 1:\n",
  "The highest-performing market is",
  top_country$Country,
  "with total revenue of",
  round(top_country$Total_Revenue, 2),
  ". This indicates that this market contributes the largest share of sales revenue.\n"
)

Business Insight 1:
 The highest-performing market is United Kingdom with total revenue of 7868409 . This indicates that this market contributes the largest share of sales revenue.


In [64]:
top_product <- top_5_products[1, ]

cat(
  "Business Insight 2:\n",
  "The top-performing product is",
  top_product$Description,
  "with revenue of",
  round(top_product$Total_Revenue, 2),
  ". This product should be considered important for inventory and sales planning.\n"
)

Business Insight 2:
 The top-performing product is PAPER CRAFT , LITTLE BIRDIE with revenue of 168469.6 . This product should be considered important for inventory and sales planning.


In [65]:
top_customer <- top_5_customers[1, ]

cat(
  "Business Insight 3:\n",
  "The highest-value customer is CustomerID",
  top_customer$CustomerID,
  "with a total purchase value of",
  round(top_customer$Total_Purchase_Value, 2),
  ". High-value customers can be targeted with personalized offers and retention strategies.\n"
)

Business Insight 3:
 The highest-value customer is CustomerID 18102 with a total purchase value of 383153 . High-value customers can be targeted with personalized offers and retention strategies.


In [66]:
cat(
  "========== THREE BUSINESS INSIGHTS ==========\n\n",

  "1. HIGH-PERFORMING MARKET\n",
  "Country:", top_country$Country,
  "\nRevenue:", round(top_country$Total_Revenue, 2),
  "\n\n",

  "2. TOP PRODUCT\n",
  "Product:", top_product$Description,
  "\nRevenue:", round(top_product$Total_Revenue, 2),
  "\n\n",

  "3. HIGHEST-VALUE CUSTOMER\n",
  "CustomerID:", top_customer$CustomerID,
  "\nPurchase Value:", round(top_customer$Total_Purchase_Value, 2),
  "\n"
)

========== THREE BUSINESS INSIGHTS ==========

 1. HIGH-PERFORMING MARKET
 Country: United Kingdom 
Revenue: 7868409 

 2. TOP PRODUCT
 Product: PAPER CRAFT , LITTLE BIRDIE 
Revenue: 168469.6 

 3. HIGHEST-VALUE CUSTOMER
 CustomerID: 18102 
Purchase Value: 383153 


In [67]:
dbListFields(
  db,
  "retail_sales"
)

[1] "InvoiceNo"         "StockCode"         "CustomerID"       
 [4] "Quantity"          "InvoiceDate"       "Description"      
 [7] "UnitPrice"         "Country"           "Revenue"          
[10] "Customer_Category"

In [68]:
test_data <- dbGetQuery(
  db,
  "
  SELECT *
  FROM retail_sales
  LIMIT 10
  "
)

print(test_data)

   InvoiceNo StockCode CustomerID Quantity         InvoiceDate
1     536365    85123A      17850        6 2010-12-01 08:26:00
2     536365     71053      17850        6 2010-12-01 08:26:00
3     536365    84406B      17850        8 2010-12-01 08:26:00
4     536365    84029G      17850        6 2010-12-01 08:26:00
5     536365    84029E      17850        6 2010-12-01 08:26:00
6     536365     22752      17850        2 2010-12-01 08:26:00
7     536365     21730      17850        6 2010-12-01 08:26:00
8     536366     22633      17850        6 2010-12-01 08:28:00
9     536366     22632      17850        6 2010-12-01 08:28:00
10    536367     84879      13047       32 2010-12-01 08:34:00
                           Description UnitPrice        Country Revenue
1   WHITE HANGING HEART T-LIGHT HOLDER      2.55 United Kingdom   15.30
2                  WHITE METAL LANTERN      3.39 United Kingdom   20.34
3       CREAM CUPID HEARTS COAT HANGER      2.75 United Kingdom   22.00
4  KNITTED UNION FL

In [69]:
dbDisconnect(db)

print("SQLite database connection closed successfully.")

[1] "SQLite database connection closed successfully."
